# File: **energy_balance.csv**

In [ ]:
######################################## Parameters

### Run
name = 'case_heating_1'
prefix = ''

In [ ]:
##### Import packages
import os
import sys
import warnings
import pandas as pd
import matplotlib.pyplot as plt


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
warnings.filterwarnings('ignore', category=UserWarning)

Load file and show its content.

In [ ]:
df = xp.load_file_csv(
    params,
    filename='energy_balance.csv',
    location='results',
    prefix=prefix,
    name=name,
    folder='csvs',
    skiprows=4,
    header=None,
    names=['component', 'carrier', 'bus_carrier', 'value'],
)

df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df.dropna(subset=['value'])
df.head()

## bar summary

Show in a bar plot the balance per item and group.

In [ ]:
#################### Parameters

### Define groups according to bus_carrier.
dic_group = {
    'electricity': ['AC', 'DC', 'low voltage', 'battery', 'EV battery', 'home battery'],
    'H2': ['H2'],
    'gas': ['gas'],
    'heat': ['urban central heat', 'urban decentral heat', 'rural heat',
             'urban central water tanks', 'urban decentral water tanks', 'rural water tanks',
             'urban central water pits'],
    'co2': ['co2', 'co2 stored', 'co2 sequestered', 'co2 emitted'],
    'NH3': ['NH3'],
    'methanol': ['methanol'],
}


### Threshold to ignore items (key: threshold value, value: list of groups it applies to)
dic_thresholds = {
    1e6: ['electricity', 'H2', 'gas', 'heat', 'co2'],  # MWh / tons
}


### Groups to plot
list_groups_plot = ['electricity', 'H2', 'gas', 'heat', 'co2']


### Unit change in plot
dic_unit_change_plot = {
    'electricity': 1e-6,  # MWh -> TWh
    'H2': 1e-6,  # MWh -> TWh
    'gas': 1e-6,  # MWh -> TWh
    'heat': 1e-6,  # MWh -> TWh
    'co2': 1e-6,  # tons -> Mt
}

dic_units_plot = {
    'electricity': 'TWh',
    'H2': 'TWh',
    'gas': 'TWh',
    'heat': 'TWh',
    'co2': 'Mt',
}

In [ ]:
### Assign group to each row based on bus_carrier

def _assign_group(bus_carrier):
    for g, carriers in dic_group.items():
        if bus_carrier in carriers:
            return g
    print(f"[Warning] bus_carrier '{bus_carrier}' not found in any group of dic_group. Assigned to 'other'.")
    return 'other'

df['group'] = df['bus_carrier'].apply(_assign_group)

df

In [ ]:
### Process pipeline per group (filter + merge Link losses + merge charger/discharger + threshold)

import re

# Invert dic_thresholds: {group: threshold}
_group_threshold = {g: thr for thr, groups in dic_thresholds.items() for g in groups}

def _charger_key(s):
    return re.sub(r'\bdischarger\b', 'charger', s)

def _strip_charger(s):
    return re.sub(r'\s*\bcharger\b\s*', ' ', s).strip()


def process_group(group):
    df_group = df[df['group'] == group].copy()

    # --- Merge Link rows sharing the same carrier into a single 'losses' row
    #
    # Rationale: in energy_balance.csv a Link with input on one bus and output on
    # another bus shows up as TWO rows with the SAME carrier — e.g. a battery
    # charger has one row on bus 'AC' (negative, input) and one on bus 'battery'
    # (positive, output). When both buses fall in the same group, the rows
    # collapse into a single bar whose value equals the net (input - |output|),
    # i.e. the conversion losses of that Link.
    #
    # However, a Link can also have multiple INPUTS (all negative) or multiple
    # OUTPUTS (all positive) within the same group — e.g. DAC consumes both
    # 'urban central heat' and 'urban decentral heat'. Those rows are NOT a
    # losses pair: their sum is the combined consumption, not a loss. Merging
    # them into '<carrier> losses' would be misleading.
    #
    # Heuristic: merge only when the values have MIXED SIGNS (at least one
    # positive and one negative). This naturally identifies the input+output
    # pattern (a true losses pair) and skips the multi-input or multi-output
    # patterns. It is a generic rule that applies to every group and avoids
    # hardcoding carrier names.
    _links = df_group[df_group['component'] == 'Link']
    _carrier_groups = _links.groupby('carrier', sort=False)
    _to_merge_link = {}  # carrier -> list of df indices to merge
    for carrier, grp in _carrier_groups:
        if len(grp) < 2:
            continue
        vals = grp['value'].values
        if (vals > 0).any() and (vals < 0).any():  # mixed signs => true losses pair
            _to_merge_link[carrier] = grp.index.tolist()

    _merged_rows = {
        carrier: {
            'component': 'Link',
            'carrier': carrier + ' losses',
            'bus_carrier': '+'.join(df_group.loc[idxs, 'bus_carrier'].values),
            'value': float(df_group.loc[idxs, 'value'].sum()),
            'group': df_group.loc[idxs[0], 'group'],
        }
        for carrier, idxs in _to_merge_link.items()
    }

    _skip_link = {i for idxs in _to_merge_link.values() for i in idxs[1:]}
    _first_to_carrier = {idxs[0]: carrier for carrier, idxs in _to_merge_link.items()}

    _rows = []
    for idx, row in df_group.iterrows():
        if idx in _skip_link:
            continue
        if idx in _first_to_carrier:
            _rows.append(_merged_rows[_first_to_carrier[idx]])
        else:
            _rows.append(row.to_dict())
    df_group = pd.DataFrame(_rows, columns=df_group.columns).reset_index(drop=True)

    # --- Combine rows whose carrier differs only in 'charger' vs 'discharger'
    df_group['_key'] = df_group['carrier'].apply(_charger_key)

    _to_merge = {}
    for name, grp in df_group.groupby(['component', 'group', '_key'], sort=False):
        if len(grp) < 2:
            continue
        carriers = grp['carrier'].tolist()
        if any('charger' in c for c in carriers) and any('discharger' in c for c in carriers):
            _to_merge[name] = grp.index.tolist()
            print(
                f"[{group}][Merge charger/discharger] component='{name[0]}' | "
                f"{carriers} -> value sum = {grp['value'].sum():.3e}"
            )

    _merged_val   = {key: df_group.loc[idxs, 'value'].sum() for key, idxs in _to_merge.items()}
    _skip         = {idx for idxs in _to_merge.values() for idx in idxs[1:]}
    _first_to_key = {idxs[0]: key for key, idxs in _to_merge.items()}

    _rows = []
    for idx, row in df_group.iterrows():
        if idx in _skip:
            continue
        d = row.to_dict()
        if idx in _first_to_key:
            key = _first_to_key[idx]
            d['value'] = _merged_val[key]
            d['carrier'] = _strip_charger(key[2])  # key = (component, group, _key)
        _rows.append(d)
    df_group = pd.DataFrame(_rows).drop(columns=['_key']).reset_index(drop=True)

    # --- Drop rows below threshold for this group (if defined)
    thr = _group_threshold.get(group)
    if thr is not None:
        _mask = df_group['value'].abs() >= thr
        _dropped = df_group[~_mask]
        if not _dropped.empty:
            print(f"[{group}][Threshold filter] {len(_dropped)} row(s) dropped (threshold={thr:g}):")
            for _, r in _dropped.iterrows():
                print(f"  - {r['component']:>10} | {r['carrier']:<40} | {r['value']:+.3e}")
        df_group = df_group[_mask].reset_index(drop=True)

    return df_group


dfs_by_group = {g: process_group(g) for g in list_groups_plot}
dfs_by_group

In [ ]:
### Vertical stack of bar charts, one row per group, sharing carrier order

# Load tech colors
plotting_cfg = xp.load_file_yaml(params, filename='plotting.default.yaml', location='config')
tech_colors = plotting_cfg['plotting']['tech_colors']
color_fallback = '#999999'

# Build master list of (component, carrier) keys in first-appearance order across groups
master_keys = []
_seen = set()
for g in list_groups_plot:
    for _, row in dfs_by_group[g].iterrows():
        key = (row['component'], row['carrier'])
        if key not in _seen:
            _seen.add(key)
            master_keys.append(key)

# Only prefix the label with the component when the same carrier appears with several components
_carrier_components = {}
for comp, carr in master_keys:
    _carrier_components.setdefault(carr, set()).add(comp)
_conflicting = {c for c, comps in _carrier_components.items() if len(comps) > 1}

labels = [
    f"{comp} {carr}" if carr in _conflicting else carr
    for comp, carr in master_keys
]
colors = [tech_colors.get(carr, color_fallback) for _, carr in master_keys]

# Per-group values reindexed to master_keys, scaled, missing filled with 0
def _values_for_group(g):
    s = dfs_by_group[g].groupby(['component', 'carrier'])['value'].sum()
    scale = dic_unit_change_plot.get(g, 1.0)
    return [float(s.get(k, 0.0)) * scale for k in master_keys]

# Precompute scaled values per group and shared y-range per unit
vals_by_group = {g: _values_for_group(g) for g in list_groups_plot}

# For each unit, take the min/max across all groups that share it, with a small symmetric pad
unit_ylim = {}
for g in list_groups_plot:
    unit = dic_units_plot.get(g, 'value')
    vals = vals_by_group[g]
    lo, hi = min(vals + [0.0]), max(vals + [0.0])
    cur_lo, cur_hi = unit_ylim.get(unit, (lo, hi))
    unit_ylim[unit] = (min(cur_lo, lo), max(cur_hi, hi))

# Add 5% padding to each unit range
unit_ylim = {
    u: (lo - 0.05 * (hi - lo) if hi != lo else lo - 1, hi + 0.05 * (hi - lo) if hi != lo else hi + 1)
    for u, (lo, hi) in unit_ylim.items()
}

n = len(list_groups_plot)
fig_width = max(8, 0.5 * len(master_keys))
fig, axes = plt.subplots(n, 1, figsize=(fig_width, 4 * n), sharex=True)
if n == 1:
    axes = [axes]

for ax, g in zip(axes, list_groups_plot):
    vals = vals_by_group[g]
    ax.bar(labels, vals, color=colors, edgecolor='black', linewidth=0.6)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(g)
    unit = dic_units_plot.get(g, 'value')
    ax.set_ylabel(unit)
    ax.set_ylim(unit_ylim[unit])
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ax.set_axisbelow(True)

axes[-1].set_xlabel('carrier')
axes[-1].tick_params(axis='x', rotation=60)
plt.setp(axes[-1].get_xticklabels(), ha='right')

plt.tight_layout()
plt.show()